In [1]:
import pandas as pd
import os
import altair as alt
from vega_datasets import data as vega_data

#df = pd.read_csv(os.path.expanduser("~/Desktop/DS4200/eia_data_with_derived_variables.csv"))
df = pd.read_csv("eia_data_with_derived_variables.csv")
df.head()

,State,Year,Coal_Consumption,CO2_Emissions,NatGas_Consumption,Nuclear_Consumption,Renewable_Consumption,Total_Energy_Consumption,Region,Dominant_Energy_Source
0,AK,1997,11719.0,589.0,425394.0,0.0,8109.0,701267.0,West,Natural Gas
1,AK,1998,16471.0,347.0,434437.0,0.0,6084.0,715652.0,West,Natural Gas
2,AK,1999,16394.0,392.0,422817.0,0.0,5040.0,719766.0,West,Natural Gas
3,AK,2000,16455.0,171.0,437972.0,0.0,5574.0,735260.0,West,Natural Gas
4,AK,2001,15911.0,465.0,413049.0,0.0,8119.0,726418.0,West,Natural Gas


In [11]:
# CO2 Emissions by Region and Energy Source
df_box = df.dropna(subset=["Region", "Dominant_Energy_Source", "CO2_Emissions"]).copy()

df_box["Q1"]     = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform(lambda x: x.quantile(0.25))
df_box["Median"] = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("median")
df_box["Q3"]     = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform(lambda x: x.quantile(0.75))
df_box["IQR"]    = df_box["Q3"] - df_box["Q1"]
df_box["Min"]    = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("min")
df_box["Max"]    = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("max")

# Darker distinct shades of green
color_scale = alt.Scale(
    domain=["Midwest", "Northeast", "South", "West"],
    range=["#0d3b24", "#1a5c38", "#2d7a4f", "#4a9a6e"]
)

base = alt.Chart(df_box)

boxes = base.mark_boxplot(
    extent=1.5,
    size=35,
    outliers=alt.MarkConfig(size=10, opacity=0.3),
).encode(
    x=alt.X("Region:N", title=None,
            axis=alt.Axis(labelAngle=-30, labelFontSize=10)),
    y=alt.Y("CO2_Emissions:Q", title="CO\u2082 Emissions (metric tons)"),
    color=alt.Color("Region:N", title="Region",
                    scale=color_scale,
                    legend=alt.Legend(orient="right")),
)

tooltip_layer = base.mark_point(opacity=0, size=900).encode(
    x=alt.X("Region:N"),
    y=alt.Y("Median:Q"),
    color=alt.Color("Region:N", scale=color_scale, legend=None),
    tooltip=[
        alt.Tooltip("Dominant_Energy_Source:N", title="Energy Source"),
        alt.Tooltip("Region:N",                 title="Region"),
        alt.Tooltip("Median:Q", title="Median CO\u2082",  format=",.0f"),
        alt.Tooltip("Q1:Q",     title="Q1 (25th %)", format=",.0f"),
        alt.Tooltip("Q3:Q",     title="Q3 (75th %)", format=",.0f"),
        alt.Tooltip("IQR:Q",    title="IQR",         format=",.0f"),
        alt.Tooltip("Min:Q",    title="Min CO\u2082",     format=",.0f"),
        alt.Tooltip("Max:Q",    title="Max CO\u2082",     format=",.0f"),
    ],
)

boxplot = (
    alt.layer(boxes, tooltip_layer)
    .properties(width=180)
    .facet(
        column=alt.Column(
            "Dominant_Energy_Source:N",
            title="Dominant Energy Source",
            header=alt.Header(labelFontSize=13)
        ),
        spacing=30
    )
    .properties(
        title=alt.TitleParams(
            text="CO\u2082 Emissions by Energy Source and Region",
            subtitle="Hover over a box to see median, IQR, and range \u00b7 Colors consistent with other figures",
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor="#6b6560"
        )
    )
    .configure_view(strokeWidth=0)
    .resolve_scale(y="shared")
)

boxplot

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.FacetChart(...)

In [13]:
import pandas as pd
import altair as alt

# Load and filter
df = pd.read_csv('eia_data.csv')
df = df[df['Year'] >= 1981]
df = df[df['State'] != 'US']

# Region mapping
region_mapping = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South',
    'SC': 'South', 'VA': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South',
    'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South',
    'TX': 'South', 'DC': 'South',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest',
    'ND': 'Midwest', 'SD': 'Midwest',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West',
    'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West',
    'HI': 'West', 'OR': 'West', 'WA': 'West'
}
df['Region'] = df['State'].map(region_mapping)

def get_dominant_energy(row):
    sources = {
        'Coal': row['Coal_Consumption'],
        'Natural Gas': row['NatGas_Consumption'],
        'Nuclear': row['Nuclear_Consumption'],
        'Renewables': row['Renewable_Consumption']
    }
    return max(sources, key=sources.get)

df['Dominant_Energy_Source'] = df.apply(get_dominant_energy, axis=1)

# Aggregate per state
state_avg = (
    df.groupby("State", as_index=False)
    .agg(
        Avg_CO2=("CO2_Emissions", "mean"),
        Avg_Total_Energy=("Total_Energy_Consumption", "mean"),
        Dominant_Energy_Source=("Dominant_Energy_Source", lambda x: x.mode()[0]),
        Region=("Region", "first"),
    )
)

# Emissions intensity: metric tons CO2 per billion Btu
state_avg["Emissions_Intensity"] = state_avg["Avg_CO2"] / state_avg["Avg_Total_Energy"] * 1e6

# FIPS lookup
fips_lookup = {
    "AL":1,"AK":2,"AZ":4,"AR":5,"CA":6,"CO":8,"CT":9,"DE":10,"FL":12,"GA":13,
    "HI":15,"ID":16,"IL":17,"IN":18,"IA":19,"KS":20,"KY":21,"LA":22,"ME":23,
    "MD":24,"MA":25,"MI":26,"MN":27,"MS":28,"MO":29,"MT":30,"NE":31,"NV":32,
    "NH":33,"NJ":34,"NM":35,"NY":36,"NC":37,"ND":38,"OH":39,"OK":40,"OR":41,
    "PA":42,"RI":44,"SC":45,"SD":46,"TN":47,"TX":48,"UT":49,"VT":50,"VA":51,
    "WA":53,"WV":54,"WI":55,"WY":56,"DC":11,
}
state_avg["id"] = state_avg["State"].map(fips_lookup)
state_avg = state_avg.dropna(subset=["id"])
state_avg["id"] = state_avg["id"].astype(int)

states_topo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/vega-datasets@1/data/us-10m.json",
    "states"
)

choropleth = (
    alt.Chart(states_topo)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(state_avg, "id", [
            "State", "Avg_CO2", "Avg_Total_Energy",
            "Emissions_Intensity", "Dominant_Energy_Source", "Region"
        ])
    )
    .encode(
        color=alt.Color(
            "Emissions_Intensity:Q",
            scale=alt.Scale(scheme="greens"),
            title="CO₂ / billion Btu",
        ),
        tooltip=[
            alt.Tooltip("State:N",               title="State"),
            alt.Tooltip("Region:N",              title="Region"),
            alt.Tooltip("Emissions_Intensity:Q", title="Intensity (mt / bil. Btu)", format=".2f"),
            alt.Tooltip("Avg_CO2:Q",             title="Avg CO₂ (metric tons)",     format=",.0f"),
            alt.Tooltip("Avg_Total_Energy:Q",    title="Avg Energy (bil. Btu)",      format=",.0f"),
            alt.Tooltip("Dominant_Energy_Source:N", title="Dominant Source"),
        ],
    )
    .project("albersUsa")
    .properties(
        width=650,
        height=400,
        title=alt.TitleParams(
            text="CO₂ Emissions Intensity by State (1981–2023)",
            subtitle="Metric tons of CO₂ per billion Btu consumed · Hover for details",
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor="#6b6560",
        ),
    )
)

choropleth


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.Chart(...)

In [ ]:
choropleth.save("choropleth.html")
boxplot.save("boxplot.html")